# Yoruba HealthQA -- Demo-Only Training (Google Colab, free GPU)

**Read this before running anything.**

This notebook trains a small proof-of-concept model so the Yoruba HealthQA app can show real generated answers during a defence/presentation. It is **not** the dissertation's final, validated model:

- The dataset used here (208 records) has been **post-edited by a Yoruba speaker** but has **NOT yet gone through clinical or linguistic validation sign-off**.
- 169 examples after safety filtering is a *very* small amount of training data. The resulting model will be undertrained and its answers should not be trusted as medically accurate.
- The base model chosen here (`Qwen2.5-1.5B-Instruct`) is picked for speed and zero licence friction on a free GPU -- it is **not** the model the dissertation's own tokeniser-fertility analysis (`scripts/07_fertility.py`) would necessarily select.

**When presenting this, say so explicitly**: *"This is a proof-of-concept fine-tune on a small, not-yet-clinically-validated batch, to demonstrate the pipeline works end to end. The dissertation's full model requires completing validation on the remaining data and training at full scale."*

## How to use this notebook
1. In Colab: **Runtime -> Change runtime type -> T4 GPU** (free tier), then **Runtime -> Run all**.
2. When prompted, upload your three dataset files (`train.jsonl`, `val.jsonl`, `test.jsonl` from your local `data/final_demo/` folder).
3. Wait for training to finish (a few minutes for this dataset size on a T4).
4. Download the zip file this notebook produces at the end.
5. On your own laptop, unzip it into your project's `models/` folder, then run the app with the two environment variables shown at the bottom of this notebook.

In [ ]:
# Confirm a GPU is actually attached. If this errors, go to
# Runtime -> Change runtime type -> T4 GPU, then re-run.
!nvidia-smi

In [ ]:
# Get the pipeline CODE (not the data) from GitHub.
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
# Install only what's needed for training, on top of Colab's pre-installed
# torch+CUDA (deliberately NOT reinstalling torch from requirements.txt --
# that pinned version may not match Colab's CUDA build and could break GPU support).
#
# NOT pinning exact versions for transformers/peft/bitsandbytes/accelerate,
# on purpose: an earlier pinned version (bitsandbytes==0.43.3) failed on a
# real run with "Using `bitsandbytes` 4-bit quantization requires
# bitsandbytes>=0.46.1" -- Colab's base image moves forward over time and a
# pin chosen today can be stale by the time you run this. Letting pip resolve
# current mutually-compatible versions is more robust here than guessing a
# fixed set of numbers.
!pip install -q -U transformers peft bitsandbytes accelerate datasets sacrebleu sentencepiece pyyaml

In [ ]:
# Upload your three dataset files (from your laptop's data/final_demo/ folder).
import os
from google.colab import files

os.makedirs("data/final_demo", exist_ok=True)
print("Select train.jsonl, val.jsonl, and test.jsonl (you can select all three at once):")
uploaded = files.upload()
for fname in uploaded:
    os.replace(fname, f"data/final_demo/{fname}")
print("Uploaded:", os.listdir("data/final_demo"))

In [ ]:
# Pick the base model and write it into configs/train.yaml.
# Qwen2.5-1.5B-Instruct: small (fast on a free GPU), no license gate to accept,
# reasonable multilingual tokenizer coverage. This is a pragmatic choice for
# a quick demo -- NOT the dissertation's methodology-selected base model.
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

import yaml

with open("configs/train.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["base_model"] = BASE_MODEL
with open("configs/train.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False, allow_unicode=True)
print(f"configs/train.yaml base_model set to {BASE_MODEL}")

In [ ]:
# Train. This is the real QLoRA fine-tune -- expect a few minutes for this
# dataset size (169 training examples) on a T4.
!python scripts/09_train.py --single-run --data-dir data/final_demo \
    --train-config configs/train.yaml --model-config configs/model.yaml --models-dir models

In [ ]:
# Package the trained adapter for download.
import shutil

shutil.make_archive("single_run_adapter", "zip", "models/single_run")
files.download("single_run_adapter.zip")

## On your own laptop, after downloading

1. Unzip `single_run_adapter.zip` into your project folder as `models/single_run/`.
2. Set these two environment variables (or put them in your `.env` file) before starting the app:

```
YHQA_BASE_MODEL=Qwen/Qwen2.5-1.5B-Instruct
YHQA_ADAPTER_PATH=models/single_run
```

3. Run the app as usual (`python app/app.py` or `docker compose up --build` with those variables set).
4. The status line in the app should now show the green "answer engine is working" message instead of the yellow placeholder one.

**A note on speed:** the first time you ask a question, the app will download the ~1.5B-parameter base model (a few GB) from Hugging Face -- this only happens once. Generating each answer on a CPU-only laptop will take longer than on the GPU used to train (maybe 10-30 seconds per answer) -- that is expected and fine for a live demo.